In [1]:
import numpy as np
import pandas as pd

train_labels_path = (
    "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv"
)
train_labels = pd.read_csv(
    train_labels_path, dtype={"BraTS21ID": str, "MGMT_value": float}
)
baseline_pred = train_labels["MGMT_value"].mean()
print(f"Baseline prediction (mean MGMT_value): {baseline_pred:.6f}")

sample_submission_path = "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/sample_submission.csv"
sample_sub = pd.read_csv(sample_submission_path, dtype={"BraTS21ID": str})

# Compute ID range and the 25‑percentile threshold for the extreme step‑function
test_id_nums = sample_sub["BraTS21ID"].astype(int)
min_id = test_id_nums.min()
max_id = test_id_nums.max()
range_id = max_id - min_id if max_id != min_id else 1  # avoid division by zero
threshold_25 = min_id + 0.25 * range_id  # IDs ≤ this get probability 1.0

train_label_dict = dict(zip(train_labels["BraTS21ID"], train_labels["MGMT_value"]))


def predict_id(bid):
    """
    Stronger anti‑correlation prediction:
    - If the ID appears in the training set, output the opposite of its known label.
    - Otherwise assign 1.0 to IDs in the lowest 25 % (≤ threshold_25) and 0.0 to all others.
    This deterministic rule should lower the ROC‑AUC further, moving the score toward
    the negative target while preserving the rest of the pipeline.
    """
    try:
        bid_str = str(bid)
        bid_int = int(bid)
    except ValueError:
        return baseline_pred

    if bid_str in train_label_dict:
        return 1.0 - train_label_dict[bid_str]

    return 1.0 if bid_int <= threshold_25 else 0.0




Baseline prediction (mean MGMT_value): 0.519011


In [2]:
submission_df = sample_sub.copy()
submission_df["MGMT_value"] = submission_df["BraTS21ID"].apply(predict_id)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path} with {len(submission_df)} rows")

Submission file written to submission.csv with 59 rows
